In [1]:
import os
os.chdir('application')

In [2]:
from ingest.langchain_milvus_ingest import store_chunks_to_milvus

C:\Users\USER\miniconda3\envs\rag-chatbot\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-23 16:46:33,628 - INFO - sentence_transformers.SentenceTransformer - sid=- - docs=- - Load pretrained SentenceTransformer: LazarusNLP/all-indo-e5-small-v4


In [3]:
from ingest.chunking import build_chunks_from_pages
from ingest.pdf_with_tables import extract_pdf_content

In [4]:
from pymilvus import connections, utility, Collection

In [5]:
from config import semantic_embedding_model_name, google_api_key_name, embedding_model_name

In [6]:
from models.embbedings import get_huggingface_embeddings

In [7]:
semantic_embeddings = get_huggingface_embeddings(semantic_embedding_model_name)

2025-10-23 16:46:38,739 - INFO - sentence_transformers.SentenceTransformer - sid=- - docs=- - Load pretrained SentenceTransformer: LazarusNLP/all-indo-e5-small-v4


In [8]:
# comment

In [9]:
folder_path = '../assets'

In [10]:
all_chunks = []
for filename in os.listdir(folder_path):
    filepath = os.path.join(folder_path, filename)
    if filepath.endswith('.pdf'):
        print(filepath)
        pdf_pages = extract_pdf_content(filepath)

        filename = os.path.basename(filepath)
        chunks = build_chunks_from_pages(pdf_pages, doc_id=filename, source=filename, embeddings_for_semantic=semantic_embeddings)
        all_chunks.extend(chunks)

../assets\004 CP_Kompensasi Kelebihan Jam Kerja HO (2025).pdf
../assets\CP-Bantuan-Pengobatan-2025.pdf
../assets\CP-Cuti-dan-Izin-Khusus.pdf
../assets\CP_Perjalanan Dinas.pdf
../assets\[Controlled Copy] Bantuan Sukacita dan Dukacita - Aug2024.pdf
../assets\[DB] Alih Daya Dari Sisi Ketenagakerjaan - Jan 2025.pdf


In [10]:
all_chunks[10]

{'chunk_id': 'CP-Bantuan-Pengobatan-2025.pdf::rule::0001::0011',
 'chunk_text': 'Feb 2025 — BANTUAN RAWAT INAP adalah bantuan biaya kesehatan yang diberikan Perusahaan',
 'checksum': 'cf04737d37d541d8ad55f9d716ae077ea838f5b7',
 'source': 'CP-Bantuan-Pengobatan-2025.pdf',
 'page': 1,
 'element_type': 'policy_rule',
 'breadcrumb': 'Feb 2025',
 'list_id': None,
 'table_id': None,
 'row_id': None,
 'col_name': None}

In [11]:
for chunk in all_chunks:
#     if chunk['source']=='CP_Perjalanan Dinas.pdf' and chunk['element_type']=='table_row' and chunk['page']==25:
    if chunk['source']=='CP_Perjalanan Dinas.pdf' and chunk['page']==26:
        print(chunk['chunk_text'], '\n\n')

Dan Kelas Penerbangan Untuk Perjalanan Dinas — 1)SelainSingapore,HongKong,danJepang. 


Dan Kelas Penerbangan Untuk Perjalanan Dinas — 2)Dalamkondisi total jampenerbangan tersingkat ≥6(lebih dari sama dengan enam)jam, baik untukkeberangkatan ataukepulangan dalam1(satu) PDJJkeluar 


Dan Kelas Penerbangan Untuk Perjalanan Dinas — negeri,makaKaryawandiijinkanmenggunakankelasBusiness 


Dan Kelas Penerbangan Untuk Perjalanan Dinas — Row[Gol.=10] • Col[Gol.] ⇒ 10 


Dan Kelas Penerbangan Untuk Perjalanan Dinas — Row[Gol.=10] • Col[Uang Saku Dinas - Dalam Negeri (dalam ribuan Rupiah)] ⇒ - 


Dan Kelas Penerbangan Untuk Perjalanan Dinas — Row[Gol.=10] • Col[Uang Saku Dinas - Asia1), Afrika (dalamUSD)] ⇒ - 


Dan Kelas Penerbangan Untuk Perjalanan Dinas — Row[Gol.=10] • Col[Uang Saku Dinas - Non Asia& Afrika, Singapore, Hong Kong, dan Jepang (dalam USD)] ⇒ - 


Dan Kelas Penerbangan Untuk Perjalanan Dinas — Row[Gol.=10] • Col[Uang Makan Dinas - Dalam Negeri (dalam ribuan Rupiah) - Makan Pagi]

In [15]:
for chunk in all_chunks:
    if chunk['source']=='CP-Cuti-dan-Izin-Khusus.pdf':
        print(chunk['chunk_text'], '\n\n')

Cuti Dan Izin Khusus — DEFINISI 


Cuti Dan Izin Khusus — CUTI adalah hak untuk tidak melakukan pekerjaan dalam periode tertentu yang diberikan kepada Karyawan sesuai dengan ketentuan yang ditetapkan oleh Perusahaan dengan mempertimbangkan ketentuan perundangan yang berlaku. 


Cuti Dan Izin Khusus — KUARTAL adalah periode waktu selama 3 (tiga) bulan berturut-turut. 


Cuti Dan Izin Khusus — PERUSAHAAN adalah seluruh entitas-entitas legal yang berdasarkan keputusan pimpinan tertinggi atau direksi KPN Group termasuk dalam ruang lingkup KPN Corporation. 


Cuti Dan Izin Khusus — TANGGAL MASUK KERJA (TMK) adalah tanggal efektif karyawan mulai bekerja di Perusahaan. 


Cuti Dan Izin Khusus — IZIN KHUSUS adalah kesempatan yang diberikan Perusahaan untuk tidak melakukan pekerjaan dengan tetap menerima upah sejumlah waktu tertentu karena adanya kejadian-kejadian spesifik yang ditentukan dalam peraturan perundangan yang berlaku di Indonesia. 


Cuti Dan Izin Khusus — CUTI TAHUNAN adalah Cuti d

In [11]:
HOST = "127.0.0.1"
PORT = "19530"
COL = "company_policies"  # your collection name

# 1) connect
connections.connect(alias="default", host=HOST, port=PORT)

# 2) show collections
print("Collections:", utility.list_collections())

# 3) does collection exist?
print("Has collection:", utility.has_collection(COL))

# 4) inspect schema (if exists)
if utility.has_collection(COL):
    c = Collection(COL)
    print("Num entities:", c.num_entities)
    print("Schema fields:")
    for f in c.schema.fields:
        print("  ", f.name, f.dtype, getattr(f, "dim", None))

# 5) run a checksum query (example)
if utility.has_collection(COL):
    rows = c.query(expr='checksum in ["sha256-some"]', output_fields=["checksum","metadata"])
    print(rows)
    coll = c

Collections: ['company_policies']
Has collection: True
Num entities: 834
Schema fields:
   id 21 None
   embedding 101 3072
   checksum 21 None
   text 21 None
   metadata 23 None
data: [], extra_info: {}


In [12]:
if utility.has_collection("company_policies"):
    utility.drop_collection("company_policies")
    print("deleted")

deleted


In [13]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
api_key = os.environ.get(google_api_key_name)
# api_key

In [14]:
skipped, inserted = store_chunks_to_milvus(
    chunks=all_chunks,
    collection_name="company_policies",
    milvus_host="127.0.0.1",
    milvus_port="19530",
#     embedding_model_name=embedding_model_name,
    embed_batch_size=32,
    dedupe_with_milvus=True,
    dry_run=False
)
print("skipped, inserted =", skipped, inserted)


2025-10-23 16:47:20,275 - INFO - ingest.langchain_milvus_ingest - sid=- - docs=- - Created collection 'company_policies' (dim=3072)
2025-10-23 16:49:48,931 - INFO - ingest.langchain_milvus_ingest - sid=- - docs=- - Finished ingest: inserted=834, skipped=0


skipped, inserted = 0 834


In [ ]:
stop

In [ ]:
# assume coll exists from previous snippet
checksum = "sha256-your-checksum-here"
rows = coll.query(expr=f'checksum in ["{checksum}"]', output_fields=["id","checksum","metadata"])
print("Query by checksum ->", rows)
# or by chunk_id if you stored it as primary id:
chunk_id = "c-12345"
rows2 = coll.query(expr=f'id in ["{chunk_id}"]', output_fields=["id","checksum","metadata"])
print("Query by id ->", rows2)


In [20]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # or fallback import

In [21]:
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
# many wrappers read env automatically; if yours needs explicit key, pass google_api_key=api_key
emb = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)

In [22]:
query_text = "Berapa biaya akomodasi untuk perjalanan dinas golongan 6 ke Sumatera?"
qvec = emb.embed_documents([query_text])[0]   # a single vector (list of floats)
print("Query vector dim:", len(qvec))

Query vector dim: 768


In [ ]:
    fields = [
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=128),
        FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=dim),
        FieldSchema(name="checksum", dtype=DataType.VARCHAR, max_length=128),
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
        FieldSchema(name="metadata", dtype=DataType.JSON),
    ]

In [25]:
# coll from earlier
TOP_K = 5
# search params for HNSW/COSINE index — tune ef (higher -> higher recall, slower)
search_params = {"metric_type": "COSINE", "params": {"ef": 100}}

results = coll.search(
    data=[qvec],                 # list of query vectors (can search multiple at once)
    anns_field="embedding",      # the name of your vector field
    param=search_params,
    limit=TOP_K,
    output_fields=["id", "checksum", "metadata"]
)

# results is a list (one item per query) of "hits" lists
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
        print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)


2025-10-13 17:46:28,687 [ERROR][handler]: RPC error: [search], <MilvusException: (code=65535, message=fail to search on QueryNode 1: worker(1) query failed: parser searchRequest failed:  => vector dimension mismatch, expected vector size(byte) 12288, actual 3072. at /workspace/source/internal/core/src/query/Plan.cpp:63
)>, <Time:{'RPC start': '2025-10-13 17:46:22.615950', 'RPC error': '2025-10-13 17:46:28.687369'}> (decorators.py:140)


MilvusException: <MilvusException: (code=65535, message=fail to search on QueryNode 1: worker(1) query failed: parser searchRequest failed:  => vector dimension mismatch, expected vector size(byte) 12288, actual 3072. at /workspace/source/internal/core/src/query/Plan.cpp:63
)>

In [ ]:
for f in c.schema.fields:
    print(f" - {f.name} dtype={f.dtype} dim={getattr(f,'dim',None)} max_length={getattr(f,'max_length',None)}")

In [ ]:
# Safe expr for VARCHAR id primary key — returns rows where id is non-empty
expr = "id != ''"
rows = c.query(expr=expr, output_fields=["id", "text", "metadata"], limit=5)
print("Sample rows (up to 5):")
for r in rows:
    print(r['text'], '\n\n')

In [ ]:
# Safe expr for VARCHAR id primary key — returns rows where id is non-empty
expr = "id != ''"
rows = c.query(expr=expr, output_fields=["id", "text", "metadata"], limit=5)
print("Sample rows (up to 5):")
for r in rows:
    print(r, '\n\n')

In [ ]:
for key, val in r.items():
    print(key)
    if type(val)==dict:
        for k, v in val.items():
            print('\t', k)

In [ ]:
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
#         print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)
        print(hit, '\n')


In [ ]:
# Query all chunks from page 3
rows = coll.query(
    expr="page == 3", 
    output_fields=["id", "page", "metadata"]
)
print("Results:", rows)

In [ ]:
all_chunks[-10]